# 构建Workflow Agent -天气查询助手

本文档通过构建一个天气查询助手Agent示例，演示如何使用openJiuwen创建完整的Workflow Agent。我们将从环境配置开始，逐步展示如何基于openJiuwen实现意图识别组件、LLM查询改写、结构化参数提取、外部API工具集成等功能，最终完成一个智能化的多步骤工作流的完整流程。

该代理能够处理用户的天气查询请求，通过多个步骤处理最终返回天气信息：
- **意图识别**：判断用户是否在查询天气
- **查询改写**：使用 LLM 将中文地名转换为英文，并格式化日期
- **参数提取**：从改写后的查询中提取地点和日期参数
- **API 调用**：通过插件组件调用天气 API
- **结果返回**：格式化并返回最终结果

## 环境准备

首先，让我们安装 openJiuwen SDK。

In [ ]:
pip install -U openjiuwen

设置访问LLM API所需的密钥和端点地址。这里以 SiliconFlow 提供的 Qwen 模型为例。

In [19]:
import os

API_BASE = os.getenv("API_BASE", "your api base")
API_KEY = os.getenv("API_KEY", "your api key")
MODEL_NAME = os.getenv("MODEL_NAME", "your model name")
MODEL_PROVIDER = os.getenv("MODEL_PROVIDER", "your model provider")
os.environ["LLM_SSL_VERIFY"] = "False"

## 创建LLM配置

先创建LLM配置，后续创建各类组件过程中，需要用LLM时就可以直接使用该配置了。

In [21]:
from openjiuwen.core.foundation.llm import ModelConfig
from openjiuwen.core.utils.llm.base import BaseModelInfo

def _create_model_config() -> ModelConfig:
    """创建模型配置"""
    return ModelConfig(
        model_provider=MODEL_PROVIDER,
        model_info=BaseModelInfo(
            model=MODEL_NAME,
            api_base=API_BASE,
            api_key=API_KEY,
            temperature=0.7,
            top_p=0.9,
            timeout=120,
        ),
    )

## 创建组件

现在让我们依次创建Agent中需要用到的各组件。每个组件都有特定的功能。

### 基础组件：开始、结束

创建工作流的开始和结束组件，以及工作流 schema。

In [22]:
from openjiuwen.core.workflow.components.start_comp import Start
from openjiuwen.core.workflow import End

def _create_start_component():
    """创建开始组件"""
    return Start({"inputs": [{"id": "query", "type": "String", "required": "true", "sourceType": "ref"}]})

def _create_end_component():
    """创建结束组件"""
    return End({"responseTemplate": "{{output}}"})

### 意图识别组件

识别用户不同意图进行业务分流是Workflow Agent中常用的业务编排方式。openJiuwen提供了IntentDetectionComponent系列API用于快速构建意图识别组件，还支持通过add_branch接口根据意图类别进行分流，使用非常方便。

我们将构建一个意图识别组件，用于检测用户是否有想要查看天气的意图。

In [23]:
from openjiuwen.core.workflow.components.intent_detection_comp import IntentDetectionComponent, IntentDetectionCompConfig

def _create_intent_detection_component() -> IntentDetectionComponent:
    """创建意图识别组件"""
    model_config = _create_model_config()
    config = IntentDetectionCompConfig(
        user_prompt="请判断用户意图",
        category_name_list=["查询某地天气"],
        model=model_config,
    )
    component = IntentDetectionComponent(config)
    # 根据意图分类结果决定下一步流向
    component.add_branch("${intent.classification_id} == 0", ["end"], "默认分支")
    component.add_branch("${intent.classification_id} == 1", ["llm"], "查询天气分支")
    return component

### LLM 组件

使用LLM组件将用户的中文查询改写为标准格式，主要功能包括：
- 将中文地名转换为英文
- 确保包含当前日期

In [24]:
from datetime import datetime
from openjiuwen.core.workflow import LLMComponent, LLMCompConfig

def build_current_date():
    current_datetime = datetime.now()
    return current_datetime.strftime("%Y-%m-%d")

def _create_llm_component() -> LLMComponent:
    """创建 LLM 组件用于查询改写"""
    model_config = _create_model_config()
    current_date = build_current_date()
    
    # 系统提示词模板
    SYSTEM_PROMPT_TEMPLATE = "你是一个query改写的AI助手。今天的日期是{}。"
    
    # 用户提示词，详细说明改写要求
    user_prompt = ("\n原始query为：{{query}}\n\n帮我改写原始query，要求：\n"
                   "1. 只把地名改为英文，其他信息保留中文；\n"
                   "2. 改写后的query必须包含当前的日期，默认日期为今天；\n"
                   "3. 日期为YYYY-MM-DD格式。")
    
    config = LLMCompConfig(
        model=model_config,
        template_content=[{"role": "user", "content": SYSTEM_PROMPT_TEMPLATE.format(current_date) + user_prompt}],
        response_format={"type": "text"},
        output_config={
            "query": {"type": "string", "description": "改写后的query", "required": True}
        },
    )
    return LLMComponent(config)

### 提问器组件

业务流程中有时需要用户确认或引导用户进一步输入澄清，这时就要用到提问器组件。

在查询天气的Agent中，让我们创建一个提问器组件，用于向用户进一步确认和澄清必需的时间和地点信息。

In [25]:
from openjiuwen.core.workflow.components.questioner_comp import QuestionerComponent, QuestionerConfig, FieldInfo

def _create_questioner_component() -> QuestionerComponent:
    """创建参数提取组件"""
    # 定义需要提取的字段
    key_fields = [
        FieldInfo(field_name="location", description="地点", required=True),
        FieldInfo(field_name="date", description="时间", required=True, default_value="today"),
    ]
    
    model_config = _create_model_config()
    config = QuestionerConfig(
        model=model_config,
        question_content="",  # 直接从响应中提取字段
        extract_fields_from_response=True,
        field_names=key_fields,
        with_chat_history=False,
    )
    return QuestionerComponent(config)

### 工具组件

创建天气查询工具组件，调用外部RESTful API服务。

**天气查询服务实现**：

先实现一个简单的天气查询Flask服务，这里给一个快速实现的例子：修改下面代码片段的api_key、url设置为实际可访问的天气查询服务的key和url。

复制下面代码片段保存为app.py，然后python app.py运行该服务后，可以通过天气查询服务的URL访问天气API。

```python
from flask import Flask, request, jsonify
import requests
from pypinyin import lazy_pinyin

def transform_english_name(city):
    try:
        return "".join(lazy_pinyin(city))
    except Exception as e:
        import traceback
        result = traceback.format_exc()
        print(f"异常: {result}")
        return "hangzhou"

app = Flask(__name__)


@app.route("/weather", methods=["GET"])
def weather():
    """
    查询实时天气
    调用示例：
        GET /weather?city=北京
        GET /weather?lat=39.9&lon=116.3
    """
    api_key = "your api key"
    url = "your weather service url"

    # 优先按城市名查，其次按坐标
    city = request.args.get("location")
    lat  = request.args.get("lat", type=float)
    lon  = request.args.get("lon", type=float)

    if city:
        # 将中文城市名转换为拼音
        city_pinyin = transform_english_name(city)
        params = {"q": city_pinyin, "appid": api_key, "units": "metric", "lang": "zh_cn"}
    elif lat is not None and lon is not None:
        params = {"lat": lat, "lon": lon, "appid": api_key, "units": "metric", "lang": "zh_cn"}
    else:
        return jsonify(error="请提供 ?city=城市名 或 ?lat=纬度&lon=经度"), 400

    try:
        print(f"正在查询天气: {city if city else f'坐标({lat}, {lon})'}")
        print(f"API URL: {url}")
        print(f"参数: {params}")
        r = requests.get(url, params=params, timeout=15)  # 增加超时时间到15秒
        r.raise_for_status()
        data = r.json()
    except requests.exceptions.Timeout:
        return jsonify(error="连接OpenWeatherMap API超时，请检查网络连接"), 504
    except requests.exceptions.HTTPError as e:
        return jsonify(error=f"OpenWeatherMap API错误: {e.response.status_code} - {e.response.text}"), 502
    except Exception as e:
        return jsonify(error=f"请求失败: {str(e)}"), 500

    # 按需裁剪返回字段
    result = {
        "city": data["name"],
        "country": data["sys"]["country"],
        "weather": data["weather"][0]["description"],
        "temperature": data["main"]["temp"],
        "feels_like": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "wind_speed": data["wind"]["speed"],
    }
    print(result)

    return jsonify(result)

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=9000, debug=True)

```

下面用openJiuwen框架来构建RestfulAPI工具组件

In [ ]:
from openjiuwen.core.workflow.components.tool_comp import ToolComponent, ToolComponentConfig
from openjiuwen.core.foundation.tool import RestfulApi, RestfulApiCard

def _create_plugin_component() -> ToolComponent:
    """创建天气查询插件组件"""
    tool_config = ToolComponentConfig()
    
    # 定义天气查询 RESTful API 工具
    weather_tool = RestfulApi(
        card=RestfulApiCard(
            name="WeatherReporter",
            description="天气查询插件",
            parameters={
                "type": "object",
                "properties": {
                    "location": {"description": "地点", "type": "string"},
                    "date": {"description": "日期", "type": "string"},
                },
                "required": ["location", "date"],
            },
            path="your weather service url",
            headers={},
            method="GET",
        )
    )
    
    return ToolComponent(tool_config).bind_tool(weather_tool)

## 工作流组装

现在我们将所有组件组装成一个完整的工作流。

### 工作流实例化

首先创建工作流的基础配置和 Workflow 对象。

In [27]:
from openjiuwen.core.workflow import Workflow
from openjiuwen.core.workflow import WorkflowConfig, WorkflowMetadata
from openjiuwen.agent.common.schema import WorkflowSchema
from openjiuwen.agent.workflow_agent.workflow_agent import WorkflowAgent
from openjiuwen.agent.config.workflow_config import WorkflowAgentConfig

# 工作流配置
workflow_id = "test_weather_agent"
workflow_version = "1.0"
workflow_name = "weather"

workflow_config = WorkflowConfig(
    metadata=WorkflowMetadata(
        name=workflow_name,
        id=workflow_id,
        version=workflow_version,
    )
)

# 创建工作流对象
flow = Workflow(workflow_config=workflow_config)

print(f"创建工作流: {workflow_id} v{workflow_version}")

创建工作流: test_weather_agent v1.0


### 定义并连接各组件

按业务流程编排工作流。

In [28]:
# 实例化所有组件
start = _create_start_component()
intent = _create_intent_detection_component()
llm = _create_llm_component()
questioner = _create_questioner_component()
plugin = _create_plugin_component()
end = _create_end_component()

# 注册组件到工作流
flow.set_start_comp("start", start, inputs_schema={"query": "${query}"})

# 意图识别组件接收来自 start 组件的查询
flow.add_workflow_comp("intent", intent, inputs_schema={"query": "${start.query}"})

# LLM 组件也接收来自 start 组件的原始查询（与意图识别并行）
flow.add_workflow_comp("llm", llm, inputs_schema={"query": "${start.query}"})

# 参数提取组件接收改写后的查询
flow.add_workflow_comp("questioner", questioner, inputs_schema={"query": "${llm.query}"})

# 插件组件接收提取的地点和日期参数
flow.add_workflow_comp("plugin", plugin, inputs_schema={
    "location": "${questioner.location}",
    "date": "${questioner.date}",
})

# 结束组件接收插件调用的结果
flow.set_end_comp("end", end, inputs_schema={"output": "${plugin.data}"})

# 连接工作流拓扑
flow.add_connection("start", "intent")     # start -> intent
flow.add_connection("llm", "questioner")   # llm -> questioner
flow.add_connection("questioner", "plugin") # questioner -> plugin
flow.add_connection("plugin", "end")       # plugin -> end

print("工作流拓扑连接完成")

工作流拓扑连接完成


## 创建Agent并与Workflow绑定

创建 WorkflowAgent 并绑定工作流。

In [29]:
# 创建 WorkflowAgent

def _create_workflow_schema(id, name: str, version: str) -> WorkflowSchema:
    return WorkflowSchema(id=id,
                          name=name,
                          description="天气查询工作流",
                          version=version,
                          inputs={"query": {
                              "type": "string",
                          }})

schema = _create_workflow_schema(workflow_id, workflow_name, workflow_version)
workflow_agent_config = WorkflowAgentConfig(
    id="test_weather_agent",
    version="0.1.0",
    description="测试用天气 agent",
    workflows=[schema]
)

# 创建Agent实例
workflow_agent = WorkflowAgent(workflow_agent_config)

# 绑定工作流到Agent
workflow_agent.bind_workflows([flow])

print("WorkflowAgent 创建完成并绑定工作流")

WorkflowAgent 创建完成并绑定工作流


## 工作流测试

现在我们来测试工作流代理的功能。

### 单个测试用例

首先测试一个简单的天气查询。

In [30]:
async def run_test(query):
    result = await workflow_agent.invoke({"query": query, "conversation_id": "c123"})

    print(result)

运行测试：

In [ ]:
await run_test("查询上海的天气")

## 总结

本 notebook 演示了如何使用 openJiuwen 构建一个完整的天气查询工作流代理。

### 主要功能特点：

1. **模块化设计**: 每个功能都封装在独立的组件中，便于维护和扩展
2. **意图识别**: 能够判断用户输入是否为天气查询，避免误触发
3. **智能查询改写**: 自动将中文地名转换为英文，并添加当前日期
4. **参数提取**: 从自然语言中准确提取地点和日期信息
5. **API 集成**: 通过插件组件轻松集成外部服务

### 工作流执行流程：

```
用户查询 → 意图识别 → 查询改写 → 参数提取 → API调用 → 结果返回
    ↓           ↓       ↓           ↓          ↓          ↓
  start     intent     llm     questioner    plugin      end
```

### 扩展建议：

- 可以添加更多天气相关的查询类型（如温度、湿度、风速等）
- 可以集成多个天气 API 服务作为备选方案
- 可以添加缓存机制提高性能
- 可以增加错误处理和重试机制

这个工作流框架可以轻松扩展到其他类型的查询任务，只需要修改相应的组件配置即可。